# COMP702 Final Project Notebook

This is the final cleaned notebook for my COMP702 project on comparing supervised learning models for cardiovascular disease classification.

I removed the trial-and-error cells from the development notebook and kept the parts that are actually needed to show how the final results were produced. The notebook covers the main workflow from loading and cleaning the data through to model training, evaluation and the additional analysis used in the dissertation.


## Portfolio packaging note — November 2026
This is the final analysis notebook recovered from the MSc project files. It is supplied without stored outputs. The historical comparison table is in `results/final_model_comparison.csv`, transcribed from Table 3.2 / Appendix A and cross-checked against the original working notebook.
Run `python scripts/prepare_data.py --input /path/to/cardio_train.csv` first. Data files are not redistributed. The supplied comparison figures are reported dissertation results, not a claim that this notebook has been fully rerun in this environment. Dependency versions used in the original experiment were not locked.
The additional ablation and threshold experiments are separate from the primary held-out comparison. No clinical-use claim is made.


## 1. Imports and file paths

First I load the libraries used throughout the project and set up the folders for the datasets, saved results and trained models.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency, mannwhitneyu
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate,
    GridSearchCV, RandomizedSearchCV, cross_val_predict
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Work from the project root when the notebook is inside the notebooks folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
MODEL_DIR = PROJECT_ROOT / "saved_models"

RESULTS_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)


## 2. Load the datasets

I use both the original Kaggle dataset and the cleaned dataset used for the final experiments. A few simple checks are included here to make sure the correct files have been loaded.


In [ ]:
raw = pd.read_csv(DATA_DIR / "cardio_train.csv", sep=";")
df = pd.read_csv(DATA_DIR / "cardio_final.csv")

print("Raw shape:", raw.shape)
print("Cleaned shape:", df.shape)
print("\nCleaned columns:")
print(df.columns.tolist())

assert raw.shape[0] == 70000
assert df.shape[0] == 68627
assert set(df["cardio"].unique()) == {0, 1}


## 3. Data cleaning

This section shows the cleaning steps used to go from the original file to the final dataset.

The rules are the same as the ones described in the dissertation. I used fixed plausibility checks rather than manually changing individual rows, so the process can be repeated from the original data.


In [ ]:
check = raw.copy()

# Remove the ID column and convert age from days to years
check = check.drop(columns=["id"])
check["age_years"] = check["age"] / 365.25
check = check.drop(columns=["age"])

# Apply the same plausibility checks used in the final cleaning stage
check = check[
    check["height"].between(100, 230)
    & check["weight"].between(30, 250)
    & check["ap_hi"].between(70, 300)
    & check["ap_lo"].between(40, 200)
    & (check["ap_hi"] > check["ap_lo"])
].copy()

# Create BMI and pulse pressure
check["BMI"] = check["weight"] / ((check["height"] / 100) ** 2)
check["pulse_pressure"] = check["ap_hi"] - check["ap_lo"]

# Remove extreme BMI values after calculation
check = check[check["BMI"].between(10, 70)].copy()

# Keep the same column order as the saved cleaned dataset
final_columns = [
    "age_years", "gender", "height", "weight", "BMI",
    "ap_hi", "ap_lo", "pulse_pressure",
    "cholesterol", "gluc", "smoke", "alco", "active", "cardio"
]
check = check[final_columns].reset_index(drop=True)

print("Recreated cleaned shape:", check.shape)
print("Rows removed:", len(raw) - len(check))
print("Retained proportion:", round(len(check) / len(raw) * 100, 4), "%")

assert len(check) == 68627


## 4. Initial exploration

Before modelling, I checked the cleaned dataset for missing values, class balance and the main relationships between variables.


In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing"))

print("\nTarget counts:")
display(df["cardio"].value_counts().sort_index().to_frame("count"))

print("\nTarget percentages:")
display((df["cardio"].value_counts(normalize=True).sort_index() * 100).round(2).to_frame("percent"))


In [ ]:
plt.figure(figsize=(6, 4))
counts = df["cardio"].value_counts().sort_index()
plt.bar(["No disease (0)", "Disease (1)"], counts.values)
plt.ylabel("Cleaned observations")
plt.title("Target-class distribution after data cleaning")
plt.tight_layout()
plt.show()


In [ ]:
target_corr = (
    df.corr(numeric_only=True)["cardio"]
      .drop("cardio")
      .sort_values(ascending=False)
)
display(target_corr.to_frame("Correlation with cardio"))


## 5. Statistical checks and feature redundancy

I also looked at correlation, VIF and simple association tests because some of the variables are closely related. This was mainly to understand whether engineered features such as BMI and pulse pressure were adding new information or mostly repeating information already present in the source variables.


In [ ]:
selected_corr_features = [
    "ap_hi", "ap_lo", "pulse_pressure",
    "BMI", "weight", "height", "age_years"
]
display(df[selected_corr_features].corr().round(3))


In [ ]:
vif_features = ["age_years", "height", "weight", "BMI", "ap_hi", "ap_lo", "pulse_pressure"]
X_vif = add_constant(df[vif_features])

vif_table = pd.DataFrame({
    "Feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
display(vif_table)


In [ ]:
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix, correction=False)[0]
    n = confusion_matrix.sum()
    r, k = confusion_matrix.shape
    return np.sqrt((chi2 / n) / max(min(k - 1, r - 1), 1))

categorical_features = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]
cat_results = []

for feature in categorical_features:
    table = pd.crosstab(df[feature], df["cardio"])
    chi2, p, _, _ = chi2_contingency(table)
    cat_results.append({
        "Feature": feature,
        "Chi-square": chi2,
        "p-value": p,
        "Cramers_V": cramers_v(table.values)
    })

categorical_analysis = pd.DataFrame(cat_results)
display(categorical_analysis.round(4))


In [ ]:
numerical_features_stats = ["age_years", "height", "weight", "BMI", "ap_hi", "ap_lo", "pulse_pressure"]
num_results = []

for feature in numerical_features_stats:
    g0 = df.loc[df["cardio"] == 0, feature]
    g1 = df.loc[df["cardio"] == 1, feature]
    u, p = mannwhitneyu(g0, g1, alternative="two-sided")
    # rank-biserial effect size using U statistic
    rbc = 1 - (2 * u) / (len(g0) * len(g1))
    num_results.append({
        "Feature": feature,
        "Class0_mean": g0.mean(),
        "Class1_mean": g1.mean(),
        "Mann_Whitney_U": u,
        "p_value": p,
        "Rank_biserial": rbc
    })

numerical_analysis = pd.DataFrame(num_results)
display(numerical_analysis.round(4))


## 6. Final modelling data

The final model comparison uses the ten features listed below. The same train/test split is reused for all seven classifiers so that the comparison stays consistent.


In [ ]:
target = "cardio"

final_features = [
    "age_years", "gender", "BMI", "ap_hi", "ap_lo",
    "cholesterol", "gluc", "smoke", "alco", "active"
]

X = df[final_features].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Training target distribution:")
display(y_train.value_counts(normalize=True).sort_index().round(4))


## 7. Preprocessing and cross-validation

The numerical variables are standardised inside the pipeline. The categorical/discrete variables are passed through without scaling.

For model development I used five-fold Stratified K-Fold cross-validation with `random_state=42`.


In [ ]:
numerical_features = ["age_years", "BMI", "ap_hi", "ap_lo"]
categorical_features = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", "passthrough", categorical_features)
    ],
    remainder="drop"
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}


## 8. Models used in the final comparison

These are the seven classifiers used in the dissertation. The Random Forest settings are the confirmed tuned values from the project. The KNN settings below are the tuned configuration used in the final comparison.


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(random_state=42))
    ]),
    "Random Forest (Tuned)": Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=8,
            max_features="log2",
            random_state=42,
            n_jobs=-1
        ))
    ]),
    "AdaBoost": Pipeline([
        ("preprocessor", preprocessor),
        ("model", AdaBoostClassifier(random_state=42))
    ]),
    "Linear SVM": Pipeline([
        ("preprocessor", preprocessor),
        ("model", LinearSVC(random_state=42))
    ]),
    "KNN (Tuned)": Pipeline([
        ("preprocessor", preprocessor),
        ("model", KNeighborsClassifier(
            n_neighbors=41,
            weights="uniform",
            p=1,
            n_jobs=-1
        ))
    ]),
    "Gaussian Naive Bayes": Pipeline([
        ("preprocessor", preprocessor),
        ("model", GaussianNB())
    ])
}


## 9. Cross-validation results

This cell runs the common five-fold validation setup for the seven models. It is useful for checking generalisation and comparing training and validation behaviour before looking at the final held-out test set.


In [ ]:
cv_rows = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=True
    )
    row = {"Model": name}
    for metric in scoring:
        row[f"CV_{metric}"] = scores[f"test_{metric}"].mean()
        row[f"Train_{metric}"] = scores[f"train_{metric}"].mean()
    cv_rows.append(row)

cv_summary = pd.DataFrame(cv_rows)
display(cv_summary.round(4))


## 10. Final test-set evaluation

After the model-development stage, each final model is fitted on the training partition and evaluated on the same held-out test set.


In [ ]:
final_rows = []
predictions = {}
scores_for_auc = {}
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    pred = model.predict(X_test)
    predictions[name] = pred

    if hasattr(model.named_steps["model"], "predict_proba"):
        score = model.predict_proba(X_test)[:, 1]
    else:
        score = model.decision_function(X_test)

    scores_for_auc[name] = score

    final_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, score)
    })

final_results = pd.DataFrame(final_rows)
display(final_results.round(4))


In [ ]:
# Compare with the confirmed result file saved during the project
confirmed = pd.read_csv(RESULTS_DIR / "final_model_comparison.csv")

display(confirmed.round(4))

final_results.to_csv(
    RESULTS_DIR / "reproduced_model_comparison.csv",
    index=False
)


## 11. Comparison plot

This plot gives a quick visual comparison of the five main evaluation metrics across the final models.


In [ ]:
plot_df = final_results.set_index("Model")[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]]
ax = plot_df.plot(kind="bar", figsize=(12, 6))
ax.set_ylabel("Score")
ax.set_title("Held-out test performance across final classifiers")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 12. Random Forest feature importance

I used the fitted tuned Random Forest to inspect which features contributed most to the tree splits. These values are model-specific and should not be treated as causal effects.


In [ ]:
rf_pipe = fitted_models["Random Forest (Tuned)"]
rf_model = rf_pipe.named_steps["model"]

feature_names = numerical_features + categorical_features
rf_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(rf_importance.round(4))

rf_importance.to_csv(
    RESULTS_DIR / "reproduced_random_forest_feature_importance.csv",
    index=False
)


In [ ]:
top_imp = rf_importance.head(10).sort_values("Importance")
plt.figure(figsize=(8, 5))
plt.barh(top_imp["Feature"], top_imp["Importance"])
plt.xlabel("Impurity-based importance")
plt.title("Tuned Random Forest feature importance")
plt.tight_layout()
plt.show()


## 13. Logistic Regression coefficients

The Logistic Regression coefficients give a second way of looking at feature direction. The raw coefficient sizes should still be interpreted carefully because only the numerical variables were standardised.


In [ ]:
log_pipe = fitted_models["Logistic Regression"]
log_model = log_pipe.named_steps["model"]

log_coefficients = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": log_model.coef_[0]
})
log_coefficients["Absolute_Coefficient"] = log_coefficients["Coefficient"].abs()
log_coefficients = log_coefficients.sort_values("Absolute_Coefficient", ascending=False)

display(log_coefficients.round(4))

log_coefficients.to_csv(
    RESULTS_DIR / "reproduced_logistic_regression_coefficients.csv",
    index=False
)


## 14. Feature-ablation experiment

This experiment checks whether the engineered variables actually improved the Logistic Regression results.

I compared three versions of the feature set under the same cross-validation setup rather than assuming BMI or pulse pressure would automatically help.


In [ ]:
feature_sets = {
    "Height + Weight": [
        "age_years", "gender", "height", "weight", "ap_hi", "ap_lo",
        "cholesterol", "gluc", "smoke", "alco", "active"
    ],
    "BMI engineered": [
        "age_years", "gender", "BMI", "ap_hi", "ap_lo",
        "cholesterol", "gluc", "smoke", "alco", "active"
    ],
    "BMI + Pulse Pressure": [
        "age_years", "gender", "BMI", "ap_hi", "ap_lo", "pulse_pressure",
        "cholesterol", "gluc", "smoke", "alco", "active"
    ]
}

ablation_rows = []

for label, features in feature_sets.items():
    X_ab = df[features]
    y_ab = df["cardio"]

    num = [f for f in features if f in ["age_years", "height", "weight", "BMI", "ap_hi", "ap_lo", "pulse_pressure"]]
    cat = [f for f in features if f not in num]

    prep = ColumnTransformer([
        ("num", StandardScaler(), num),
        ("cat", "passthrough", cat)
    ])

    pipe = Pipeline([
        ("preprocessor", prep),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ])

    ab_scores = cross_validate(
        pipe, X_ab, y_ab,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    ablation_rows.append({
        "Feature representation": label,
        "Accuracy": ab_scores["test_accuracy"].mean(),
        "Precision": ab_scores["test_precision"].mean(),
        "Recall": ab_scores["test_recall"].mean(),
        "F1": ab_scores["test_f1"].mean(),
        "ROC-AUC": ab_scores["test_roc_auc"].mean()
    })

ablation_results = pd.DataFrame(ablation_rows)
display(ablation_results.round(4))


## 15. Tuning setup

The following search spaces document how the Random Forest and KNN tuning was approached during the project.

These cells are left commented out because rerunning the full searches is not necessary for the final submission and can take a while.


In [ ]:
# Random Forest search space used during development
rf_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, 15],
    "model__min_samples_split": [5, 10, 20],
    "model__min_samples_leaf": [2, 4, 8],
    "model__max_features": ["sqrt", "log2"]
}

rf_search = RandomizedSearchCV(
    estimator=Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=42, n_jobs=-1))
    ]),
    param_distributions=rf_param_grid,
    n_iter=12,
    scoring="roc_auc",
    cv=cv,
    random_state=42,
    n_jobs=-1
)

# Uncomment these lines if the search needs to be rerun:
# rf_search.fit(X_train, y_train)
# print(rf_search.best_params_)
# print(rf_search.best_score_)


In [ ]:
# KNN search space used during development
knn_param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31, 41],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

knn_search = GridSearchCV(
    estimator=Pipeline([
        ("preprocessor", preprocessor),
        ("model", KNeighborsClassifier(n_jobs=-1))
    ]),
    param_grid=knn_param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)

# Uncomment these lines if the search needs to be rerun:
# knn_search.fit(X_train, y_train)
# print(knn_search.best_params_)
# print(knn_search.best_score_)

# Final tuned KNN setting used in the comparison:
# n_neighbors=41, weights='uniform', p=1


## 16. Additional Random Forest threshold analysis

This was an extra experiment used in the discussion chapter. It is kept separate from the main seven-model comparison.

The threshold is selected using training-data cross-validation, not the held-out test set.


In [ ]:
advanced_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=600,
        max_depth=12,
        min_samples_split=21,
        min_samples_leaf=18,
        max_features="sqrt",
        bootstrap=True,
        max_samples=0.7,
        class_weight={0: 1.0, 1: 1.3},
        criterion="gini",
        random_state=42,
        n_jobs=-1
    ))
])

advanced_rf.fit(X_train, y_train)
test_prob = advanced_rf.predict_proba(X_test)[:, 1]

# Check the normal 0.50 decision threshold
pred_050 = (test_prob >= 0.50).astype(int)

advanced_default = {
    "Accuracy": accuracy_score(y_test, pred_050),
    "Precision": precision_score(y_test, pred_050),
    "Recall": recall_score(y_test, pred_050),
    "F1": f1_score(y_test, pred_050),
    "ROC-AUC": roc_auc_score(y_test, test_prob)
}

pd.Series(advanced_default).round(4)


In [ ]:
# Use out-of-fold probabilities from the training data only
oof_prob = cross_val_predict(
    advanced_rf,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

threshold_rows = []
for threshold in np.arange(0.35, 0.66, 0.01):
    pred = (oof_prob >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_train, pred),
        "Precision": precision_score(y_train, pred, zero_division=0),
        "Recall": recall_score(y_train, pred, zero_division=0),
        "F1": f1_score(y_train, pred, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_rows)

controlled = threshold_df[
    (threshold_df["Precision"] >= 0.72)
    & (threshold_df["Recall"] >= 0.70)
].sort_values(["F1", "Accuracy"], ascending=False)

display(controlled.head(10).round(4))

controlled_threshold = float(controlled.iloc[0]["Threshold"])
print("Controlled threshold:", controlled_threshold)


In [ ]:
controlled_pred = (test_prob >= controlled_threshold).astype(int)

controlled_test = {
    "Accuracy": accuracy_score(y_test, controlled_pred),
    "Precision": precision_score(y_test, controlled_pred, zero_division=0),
    "Recall": recall_score(y_test, controlled_pred, zero_division=0),
    "F1": f1_score(y_test, controlled_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, test_prob)
}

pd.Series(controlled_test).round(4)


## 17. Save trained models

The fitted models can be saved so that the final estimators are available without retraining them every time.


In [ ]:
import joblib

for name, model in fitted_models.items():
    safe_name = (
        name.lower()
        .replace(" ", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("-", "_")
    )
    joblib.dump(model, MODEL_DIR / f"{safe_name}.pkl")

print("Saved models to:", MODEL_DIR)


## End

The CSV files in the `results` folder are the confirmed project outputs used when writing the dissertation. This notebook is mainly included to show the final workflow in a clean and reproducible form.
